# ⚖️ RAG Pipeline on the Egyptian Civil Code (Arabic + English)

A complete Retrieval-Augmented-Generation system over the **1,093 articles** of the Egyptian Civil Code, in both Arabic and English.

---

## 🗺️ Pipeline overview — what this notebook does, end to end

### Phase A — Preparing the knowledge base (run once)

| Step | Task | What happens |
|------|------|--------------|
| 1 | **Load JSON** | Read 1,093 articles from the JSON file, extract `id` + `text` + metadata for each language |
| 2 | **Preprocess & Chunk** | Unicode-normalize, strip noise, split long articles into sentence-aware overlapping chunks (~2,262 chunks total) |
| 3 | **Embed** | Convert every chunk into a 1024-dim vector using **BGE-M3** (free multilingual model) |
| 4 | **Store** | Index the vectors into **three** vector databases: **Chroma**, **Qdrant**, and (optional) **Elasticsearch** |

### Phase B — Answering a user question (per query)

| Step | Task | What happens |
|------|------|--------------|
| 5 | **Vector Search** | Encode the question, retrieve top-k similar chunks from the chosen DB |
| ⭐ | **Reranker (bonus)** | Re-order the candidates with a cross-encoder for higher precision |
| 6 + 7 | **RAG + Prompt** | Stuff the chunks into a grounded prompt that asks the LLM to cite article numbers |
| — | **LLM** | A local **Qwen** model served via **Ollama** writes the natural-language answer with citations |
| 9 | **End-to-end pipeline** | One `rag_pipeline()` function wires it all together |

### Phase C — Measuring quality

| Step | Task | What we measure |
|------|------|-----------------|
| 10 | **Retrieval metrics** | Hit@k, Recall@k, MRR, nDCG@k on a curated gold set |
| — | **LLM-as-a-Judge** | Faithfulness / Relevance / Completeness scored 1–5 by the LLM |
| — | **Latency benchmark** | Per-stage timings: embed → search → rerank |
| — | **Backend comparison** | Same query → Chroma vs Qdrant vs Elasticsearch, side-by-side |

### Phase D — Interactive UI

A **Gradio web app** wraps everything in four tabs:
1. 💬 **Ask** — chat with the corpus (pick the vector DB, language filter, reranker on/off)
2. 🔬 **Compare backends** — same query through every backend at once
3. 📊 **Benchmark** — latency breakdown
4. 🧪 **Evaluation** — retrieval metrics + LLM-as-a-Judge

---

## 🆓 No paid services required

| Component | What we use | Cost |
|-----------|-------------|------|
| Embedding model | `BAAI/bge-m3` (Hugging Face) | Free |
| Reranker | `cross-encoder/mmarco-mMiniLMv2-L12` (Hugging Face) | Free |
| LLM | `qwen2.5:7b-instruct` (served locally via Ollama) | Free |
| Vector DBs | Chroma + Qdrant in-memory | Free |
| Optional | Elasticsearch (only if you have a cluster) | Optional |

Anthropic Claude or OpenAI GPT can be plugged in via `LLM_BACKEND` if you have keys, but the default works without any account.

---

## ▶️ How to run on Kaggle

1. Add the JSON file as a Kaggle Dataset (**+ Add Data**).
2. Set `JSON_PATH` in the **Config** cell.
3. **Settings → Internet: ON** (first-time model downloads).
4. **Settings → Accelerator: GPU T4** (optional, makes the LLM ~5× faster).
5. **Run All**.


# 📚 Phase A — Preparing the knowledge base

## 0. Setup — install dependencies & imports

We install everything up front so the rest of the notebook runs cleanly. Total install: ~2 minutes.

In [1]:
# Install everything in one clean cell. No -U upgrades — keep Kaggle's existing versions.
!pip uninstall -y -q transformers tokenizers huggingface_hub sentence-transformers 2>/dev/null
!pip install -q chromadb sentence-transformers qdrant-client elasticsearch gradio accelerate
# Ollama Python client (we chat with a local Qwen model through Ollama).
!pip install -q ollama

# Install the Ollama runtime/binary so we can serve the Qwen model locally on Kaggle.
# The new Ollama installer needs `zstd` to unpack — install it first to avoid:
#   "ERROR: This version requires zstd for extraction".
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 993.6/993.6 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# (Install moved to the cell above — this cell intentionally empty)

In [3]:
# (Install moved to the cell above — this cell intentionally empty)

In [4]:
import json
import re
import os
import unicodedata
from pathlib import Path
from typing import List, Dict, Any

import numpy as np

## ⚙️ Configuration

Edit `JSON_PATH` to point at your uploaded file.

In [5]:
# --- CONFIG ---------------------------------------------------------------
JSON_PATH = "/kaggle/input/datasets/mohammedbahgat/gen-ai-project/1576751803.json"
print("Using JSON file:", JSON_PATH)

# Embedding model — BGE-M3: a top multilingual embedding model (Arabic + English),
# free, runs locally, produces 1024-dim dense vectors.
EMBED_MODEL_NAME = "BAAI/bge-m3"

# Chunking settings (in characters — simple & language-agnostic).
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

# Where ChromaDB will persist its data.
CHROMA_DIR = "/kaggle/working/chroma_db" if os.path.exists("/kaggle/working") else "./chroma_db"
COLLECTION_NAME = "egyptian_civil_code"

# Batch size for embedding + Chroma inserts.
BATCH_SIZE = 64

# --- OLLAMA (LLM) CONFIG ---------------------------------------------------
# We chat with a local Qwen model served through Ollama.
# qwen2.5:7b-instruct is multilingual and handles Arabic + English well.
# Switch to "qwen2.5:3b-instruct" if you are tight on GPU memory (faster, lighter).
OLLAMA_MODEL = "qwen2.5:7b-instruct"
OLLAMA_HOST = "http://127.0.0.1:11434"


Using JSON file: /kaggle/input/datasets/mohammedbahgat/gen-ai-project/1576751803.json


## 🧩 Task 1 — Load JSON & extract records

The file is a dict with ~1,093 keys named `Article 1`, `Article 2`, …
Each value has three fields: `arabic`, `english`, `metadata` (a list of section/chapter labels).

We build one record per **language per article** so we can retrieve in either Arabic or English. Each record has at minimum `id` and `text`, plus useful metadata.

In [6]:
def load_records(json_path: str) -> List[Dict[str, Any]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records: List[Dict[str, Any]] = []
    for key, value in data.items():
        # Skip non-article entries like __metadata__ or __unmatched_text__
        if not key.startswith("Article"):
            continue
        if not isinstance(value, dict):
            continue

        # Article number (the digits at the end of the key).
        m = re.search(r"(\d+)", key)
        article_num = int(m.group(1)) if m else -1

        meta_list = value.get("metadata", []) or []
        meta_str = " | ".join(meta_list)  # Chroma metadata must be primitive types

        for lang_key, lang_code in [("arabic", "ar"), ("english", "en")]:
            text = (value.get(lang_key) or "").strip()
            if not text:
                continue  # skip empty translations
            records.append({
                "id": f"art{article_num}_{lang_code}",
                "article_number": article_num,
                "language": lang_code,
                "section_path": meta_str,
                "text": text,
            })
    return records

records = load_records(JSON_PATH)
print(f"Extracted {len(records)} records (one per language per article).")
print("\nSample record (English):")
print(json.dumps(next(r for r in records if r["language"] == "en"), ensure_ascii=False, indent=2)[:800])
print("\nSample record (Arabic):")
print(json.dumps(next(r for r in records if r["language"] == "ar"), ensure_ascii=False, indent=2)[:800])

Extracted 2178 records (one per language per article).

Sample record (English):
{
  "id": "art1_en",
  "article_number": 1,
  "language": "en",
  "section_path": "نصوص القانون المدنى | باب تمهيدي | أحكام عامة | الفصل الأول | القانون وتطبيقه | SECTION I | Laws and their Applications | ١ -القانون والحق 1. Laws and Rights",
  "text": "Provisions of laws govern all matters to which these \nprovisions apply in letter or spirit. \nIn the absence of a provision of a law that is applicable, \nthe Judge will decide according to custom and in the \nabsence of custom in accordance with the principles of \nMoslem Law. \nIn the absence of such principles, the Judge will apply the \nprinciples of natural justice and the rules of equity."
}

Sample record (Arabic):
{
  "id": "art1_ar",
  "article_number": 1,
  "language": "ar",
  "section_path": "نصوص القانون المدنى | باب تمهيدي | أحكام عامة | الفصل الأول | القانون وتطبيقه | SECTION I | Laws and their Applications | ١ -القانون والحق 1. Laws and Righ

## 🧹 Task 2 — Cleaning, normalization, chunking

Three things happen:

1. **Unicode normalization** (NFKC) — fixes Arabic presentation forms and stray ligatures.
2. **Whitespace & noise cleanup** — collapse multiple spaces / newlines, strip control chars and most non-essential symbols (we keep Arabic letters, Latin letters, Arabic-Indic digits, basic punctuation).
3. **Sentence-aware chunking** — most articles stay as one chunk. Long ones (e.g. Article 1143 ≈ 2.4 k chars) get split into overlapping chunks for better embedding & retrieval.

In [7]:
# Allowed character classes (we *keep* these and strip the rest).
# - \u0600-\u06FF : Arabic block (incl. tashkeel)
# - \u0660-\u0669 : Arabic-Indic digits (already in the block above, kept explicit for clarity)
# - A-Za-z0-9    : Latin letters & digits
# - basic punctuation & whitespace
_KEEP_RE = re.compile(r"[^\u0600-\u06FFA-Za-z0-9\s\.,;:?!\-\(\)\"'\u060C\u061B\u061F]")
_WS_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not text:
        return ""
    # 1. Unicode normalization
    text = unicodedata.normalize("NFKC", text)
    # 2. Drop control chars
    text = "".join(ch for ch in text if ch == "\n" or ch == "\t" or ord(ch) >= 32)
    # 3. Strip stray symbols we don't want
    text = _KEEP_RE.sub(" ", text)
    # 4. Collapse whitespace
    text = _WS_RE.sub(" ", text).strip()
    return text

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Sentence-aware sliding-window chunker.

    We try to break on sentence boundaries (., ?, !, Arabic ؟ ، ؛) so chunks read naturally.
    Falls back to hard char splits if a single 'sentence' is longer than chunk_size.
    """
    if len(text) <= chunk_size:
        return [text]

    # Split into sentences while keeping the delimiter.
    sentences = re.split(r"(?<=[\.!\?\u061F\u061B])\s+", text)
    chunks: List[str] = []
    current = ""
    for sent in sentences:
        if not sent:
            continue
        if len(current) + len(sent) + 1 <= chunk_size:
            current = (current + " " + sent).strip() if current else sent
        else:
            if current:
                chunks.append(current)
            # If a single sentence is bigger than chunk_size, hard-split it.
            if len(sent) > chunk_size:
                for i in range(0, len(sent), chunk_size - overlap):
                    chunks.append(sent[i:i + chunk_size])
                current = ""
            else:
                # Add overlap from end of previous chunk for context continuity.
                tail = current[-overlap:] if current and overlap > 0 else ""
                current = (tail + " " + sent).strip()
    if current:
        chunks.append(current)
    return chunks

# Apply cleaning + chunking to every record.
chunked_records: List[Dict[str, Any]] = []
for rec in records:
    cleaned = clean_text(rec["text"])
    if not cleaned:
        continue
    pieces = chunk_text(cleaned)
    for idx, piece in enumerate(pieces):
        chunked_records.append({
            "id": f"{rec['id']}_c{idx}",
            "text": piece,
            "article_number": rec["article_number"],
            "language": rec["language"],
            "section_path": rec["section_path"],
            "chunk_index": idx,
            "chunk_total": len(pieces),
        })

print(f"Records before chunking: {len(records)}")
print(f"Chunks after chunking:   {len(chunked_records)}")
multi = [r for r in chunked_records if r['chunk_total'] > 1]
print(f"Articles that were split: {len(set((r['article_number'], r['language']) for r in multi))}")
print("\nExample chunk:")
print(json.dumps(chunked_records[0], ensure_ascii=False, indent=2)[:600])

# ---------------------------------------------------------------------------
# SINGLE source of truth for chunk metadata. BOTH vector databases
# (Chroma + Qdrant) use THIS exact dict, so their metadata stays identical.
# ---------------------------------------------------------------------------
def build_chunk_metadata(rec: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "chunk_id": rec["id"],
        "text": rec["text"],
        "article_number": rec["article_number"],
        "language": rec["language"],
        "section_path": rec["section_path"],
        "chunk_index": rec["chunk_index"],
        "chunk_total": rec["chunk_total"],
    }


Records before chunking: 2178
Chunks after chunking:   2262
Articles that were split: 71

Example chunk:
{
  "id": "art1_ar_c0",
  "text": ")١ (تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه النصوص في لفظها أو في فحواها. (٢) فإذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضى العرف، فإذا لم يوجد، فبمقتضى مبادئ الشريعة الإسلامية، فإذا لم يوجد فبمقتضى مبادئ القانون الطبيعي وقواعد العدالة .",
  "article_number": 1,
  "language": "ar",
  "section_path": "نصوص القانون المدنى | باب تمهيدي | أحكام عامة | الفصل الأول | القانون وتطبيقه | SECTION I | Laws and their Applications | ١ -القانون والحق 1. Laws and Rights",
  "chunk_index": 0,
  "chunk_total": 1
}


## 🔢 Task 3 — Generate embeddings

We use **`BAAI/bge-m3`** through `sentence-transformers`:

- **1024-dim** dense embeddings
- a strong **multilingual** retrieval model (Arabic + English, 100+ languages)
- one of the best open embedding models — no API key, no rate limits
- ~2.2 GB; runs comfortably on GPU and also works on CPU (slower)

> We auto-detect the device: GPU if `torch.cuda.is_available()`, otherwise CPU.


In [8]:
from sentence_transformers import SentenceTransformer
import torch

# BGE-M3 is a large model — use the GPU when available, fall back to CPU otherwise.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", device)

model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

texts = [r["text"] for r in chunked_records]
embeddings = model.encode(
    texts,
    batch_size=16,           # BGE-M3 is heavier, keep batches modest
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Embeddings shape:", embeddings.shape)


Embedding device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model: BAAI/bge-m3
Embedding dimension: 1024


/tmp/ipykernel_22/3035712056.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


Batches:   0%|          | 0/142 [00:00<?, ?it/s]

Embeddings shape: (2262, 1024)


## 🗄️ Task 4 — Store in THREE vector databases

We index the **same** 2,262 chunks (with the same embeddings) into three backends so we can compare them later:

| DB | Strengths | Weaknesses |
|---|---|---|
| **Chroma** | Dead-simple, embedded, perfect for prototypes | Not made for large multi-tenant production |
| **Qdrant** | Fast, Rust-based, strong filtering, scales to billions of vectors | Extra moving piece if you're tiny |
| **Elasticsearch** | Industry-standard, **hybrid search** (BM25 keyword + vector), great for legal text | Heavier to operate, needs a running cluster |

### 4.1 Chroma — persistent local store

In [9]:
import chromadb
from chromadb.config import Settings

Path(CHROMA_DIR).mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=CHROMA_DIR, settings=Settings(anonymized_telemetry=False))

# Recreate the collection from scratch for a clean run.
if COLLECTION_NAME in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},  # cosine similarity
)

# Insert in batches (Chroma can choke on huge single inserts).
ids = [r["id"] for r in chunked_records]
# SAME builder as Qdrant -> identical metadata (chunk_id, text, article_number,
# language, section_path, ...) in BOTH vector databases.
metadatas = [build_chunk_metadata(r) for r in chunked_records]

for start in range(0, len(ids), BATCH_SIZE):
    end = start + BATCH_SIZE
    collection.add(
        ids=ids[start:end],
        documents=texts[start:end],
        metadatas=metadatas[start:end],
        embeddings=embeddings[start:end].tolist(),
    )

print(f"Inserted {collection.count()} chunks into Chroma collection '{COLLECTION_NAME}'.")
print("Chroma metadata keys:", list(metadatas[0].keys()))


Inserted 2262 chunks into Chroma collection 'egyptian_civil_code'.
Chroma metadata keys: ['chunk_id', 'text', 'article_number', 'language', 'section_path', 'chunk_index', 'chunk_total']


### 4.2 Qdrant — in-memory mode

`qdrant-client` has an `:memory:` mode that's perfect for Kaggle — no Docker, no service, just runs in the same Python process.

In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

QDRANT_COLLECTION = "egyptian_civil_code_qdrant"

qdrant = QdrantClient(":memory:")  # change to QdrantClient(url="http://localhost:6333") for a real server

# (Re)create the collection with the same vector size as our embedding model.
emb_dim = embeddings.shape[1]
if qdrant.collection_exists(QDRANT_COLLECTION):
    qdrant.delete_collection(QDRANT_COLLECTION)
qdrant.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=emb_dim, distance=Distance.COSINE),
)

# Upsert points in batches. Qdrant wants integer or UUID ids — we use the row index.
# SAME builder as Chroma -> identical metadata in BOTH vector databases.
points = []
for i, rec in enumerate(chunked_records):
    points.append(PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=build_chunk_metadata(rec),
    ))

for start in range(0, len(points), 256):
    qdrant.upsert(collection_name=QDRANT_COLLECTION, points=points[start:start + 256])

print(f"Qdrant collection populated with {qdrant.count(QDRANT_COLLECTION).count} points.")
print("Qdrant payload keys:", list(points[0].payload.keys()))


Qdrant collection populated with 2262 points.
Qdrant payload keys: ['chunk_id', 'text', 'article_number', 'language', 'section_path', 'chunk_index', 'chunk_total']


### 4.3 Elasticsearch — optional, supports hybrid search

Elasticsearch needs a running ES service. On Kaggle the easiest path is to point the client at a managed cluster (Elastic Cloud free tier, or your own) using `ES_URL` + `ES_API_KEY` Kaggle Secrets.

If you don't have a cluster, leave `ENABLE_ES = False` and the rest of the notebook will skip the ES sections. **Chroma + Qdrant alone are enough** — ES is the "production hybrid search" option.

In [11]:
ENABLE_ES = False  # set True after configuring ES_URL + ES_API_KEY secrets

es_client = None
ES_INDEX = "egyptian_civil_code_es"

if ENABLE_ES:
    from elasticsearch import Elasticsearch
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    es_client = Elasticsearch(
        secrets.get_secret("ES_URL"),
        api_key=secrets.get_secret("ES_API_KEY"),
    )

    # Recreate index with a dense_vector field + standard text field for BM25 (hybrid).
    if es_client.indices.exists(index=ES_INDEX):
        es_client.indices.delete(index=ES_INDEX)
    es_client.indices.create(
        index=ES_INDEX,
        mappings={
            "properties": {
                "text": {"type": "text"},
                "article_number": {"type": "integer"},
                "language": {"type": "keyword"},
                "section_path": {"type": "text"},
                "embedding": {
                    "type": "dense_vector",
                    "dims": emb_dim,
                    "index": True,
                    "similarity": "cosine",
                },
            }
        },
    )

    # Bulk-index. For brevity we do a simple loop; for large data use elasticsearch.helpers.bulk.
    from elasticsearch.helpers import bulk
    actions = []
    for i, rec in enumerate(chunked_records):
        actions.append({
            "_index": ES_INDEX,
            "_id": rec["id"],
            "_source": {
                "text": rec["text"],
                "article_number": rec["article_number"],
                "language": rec["language"],
                "section_path": rec["section_path"],
                "embedding": embeddings[i].tolist(),
            },
        })
    bulk(es_client, actions)
    es_client.indices.refresh(index=ES_INDEX)
    print("Elasticsearch indexed:", es_client.count(index=ES_INDEX)["count"])
else:
    print("Elasticsearch disabled (ENABLE_ES=False).")

Elasticsearch disabled (ENABLE_ES=False).


# 🔍 Phase B — Answering a user question

## 🔍 Task 5 — Vector search functions

One search function per backend, all with the same signature: `(query, k, language) -> list[hit]`. Each hit is a dict with `text`, `article_number`, `language`, `section_path`, `score`.

In [12]:
def vector_search(query: str, k: int = 5, language: str | None = None) -> list[dict]:
    """Return the top-k chunks most similar to `query` from Chroma."""
    q_emb = model.encode([query], normalize_embeddings=True).tolist()
    where = {"language": language} if language else None
    res = collection.query(query_embeddings=q_emb, n_results=k, where=where)

    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({
            "text": doc,
            "article_number": meta["article_number"],
            "language": meta["language"],
            "section_path": meta.get("section_path", ""),
            "score": 1 - dist,  # cosine distance -> similarity
        })
    return hits


# Quick smoke-test
demo = vector_search("rights of the lessee in a lease contract", k=3, language="en")
for h in demo:
    print(f"[Art {h['article_number']} | sim={h['score']:.3f}] {h['text'][:150]}...")

[Art 589 | sim=0.724] The lessor has, as warranty for all amounts due to him under the agreement of lease; a lien on all the attachable movables stocking the leased propert...
[Art 558 | sim=0.720] A lease is a contract by which the lessor undertakes to enable the lessee to enjoy a specific thing for a certain time in return of a fixed rent....
[Art 579 | sim=0.717] The lessee must use the leased property in the manner agreed. In the absence of any agreement, he must use the property in accordance with the purpose...


In [13]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def vector_search_qdrant(query: str, k: int = 5, language: str | None = None) -> list[dict]:
    q_emb = model.encode([query], normalize_embeddings=True).tolist()[0]

    flt = None
    if language:
        flt = Filter(must=[FieldCondition(key="language", match=MatchValue(value=language))])

    res = qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=q_emb,
        limit=k,
        query_filter=flt,
        with_payload=True,
    ).points

    hits = []
    for p in res:
        hits.append({
            "text": p.payload["text"],
            "article_number": p.payload["article_number"],
            "language": p.payload["language"],
            "section_path": p.payload.get("section_path", ""),
            "score": p.score,  # Qdrant returns cosine similarity directly
        })
    return hits


demo_q = vector_search_qdrant("rights of the lessee in a lease contract", k=3, language="en")
for h in demo_q:
    print(f"[Qdrant | Art {h['article_number']} | sim={h['score']:.3f}] {h['text'][:150]}...")

[Qdrant | Art 589 | sim=0.724] The lessor has, as warranty for all amounts due to him under the agreement of lease; a lien on all the attachable movables stocking the leased propert...
[Qdrant | Art 558 | sim=0.720] A lease is a contract by which the lessor undertakes to enable the lessee to enjoy a specific thing for a certain time in return of a fixed rent....
[Qdrant | Art 579 | sim=0.717] The lessee must use the leased property in the manner agreed. In the absence of any agreement, he must use the property in accordance with the purpose...


In [14]:
def vector_search_es(query: str, k: int = 5, language: str | None = None,
                     hybrid: bool = True) -> list[dict]:
    """ES search. If hybrid=True we combine BM25 (keyword) + kNN (vector). 
    Otherwise pure vector search."""
    if not ENABLE_ES or es_client is None:
        return []

    q_emb = model.encode([query], normalize_embeddings=True).tolist()[0]
    filt = [{"term": {"language": language}}] if language else []

    knn = {
        "field": "embedding",
        "query_vector": q_emb,
        "k": k,
        "num_candidates": max(50, k * 10),
        "filter": filt,
    }

    body = {"knn": knn, "size": k}
    if hybrid:
        # BM25 portion — combined automatically by ES with knn (score sum).
        body["query"] = {
            "bool": {
                "must": [{"match": {"text": query}}],
                "filter": filt,
            }
        }

    res = es_client.search(index=ES_INDEX, body=body)
    hits = []
    for h in res["hits"]["hits"]:
        src = h["_source"]
        hits.append({
            "text": src["text"],
            "article_number": src["article_number"],
            "language": src["language"],
            "section_path": src.get("section_path", ""),
            "score": h["_score"],
        })
    return hits


if ENABLE_ES:
    demo_es = vector_search_es("rights of the lessee in a lease contract", k=3, language="en")
    for h in demo_es:
        print(f"[ES | Art {h['article_number']} | score={h['score']:.3f}] {h['text'][:150]}...")
else:
    print("(skipped — ENABLE_ES is False)")

(skipped — ENABLE_ES is False)


## ⭐ Bonus — Cross-encoder reranker

### What is reranking and why bother?

The embedding model we use for retrieval is a **bi-encoder**: it encodes the query and the documents *separately*, then compares vectors. That's fast (millions of docs in milliseconds) but the comparison is shallow — just a dot product of two summary vectors.

A **cross-encoder** (the reranker) takes a *(query, document)* **pair together** and runs them through a transformer that attends across both. Much slower per pair, but vastly more accurate.

**The pattern:**
1. Bi-encoder fetches top-N (e.g. N=20) candidates — cheap and fast.
2. Cross-encoder reranks those 20 — accurate but small N keeps it fast.
3. Return top-k after reranking (e.g. k=5).

This combo is the de-facto standard for production RAG.

In [15]:
from sentence_transformers import CrossEncoder

# Multilingual cross-encoder — works for Arabic + English queries against Arabic + English docs.
RERANKER_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
reranker = CrossEncoder(RERANKER_NAME, device="cpu")
print("Reranker loaded:", RERANKER_NAME)


def vector_search_with_rerank(query: str, k: int = 5, candidate_k: int = 20,
                              language: str | None = None) -> list[dict]:
    """Retrieve `candidate_k` from Chroma, rerank, return top `k`."""
    candidates = vector_search(query, k=candidate_k, language=language)
    if not candidates:
        return []
    pairs = [(query, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)
    # Attach rerank scores and resort
    for c, s in zip(candidates, rerank_scores):
        c["rerank_score"] = float(s)
    candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
    return candidates[:k]


# Demo: compare before vs after rerank on the same query
q = "What rules apply when there is no written law?"
print("=== Before reranking (top-5 from Chroma) ===")
for h in vector_search(q, k=5, language="en"):
    print(f"  [Art {h['article_number']:>4}] sim={h['score']:.3f}  {h['text'][:120]}...")

print("\n=== After reranking (fetched 20, kept best 5) ===")
for h in vector_search_with_rerank(q, k=5, candidate_k=20, language="en"):
    print(f"  [Art {h['article_number']:>4}] rerank={h['rerank_score']:+.3f}  {h['text'][:120]}...")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Reranker loaded: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
=== Before reranking (top-5 from Chroma) ===
  [Art    1] sim=0.623  Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision...
  [Art  856] sim=0.623  In the absence of any provisions to the contrary in the...
  [Art   86] sim=0.594  Rights in respect of a non-material object are regulated by special laws....
  [Art  873] sim=0.578  Rights of fishing and hunting and rights to things found and to antiquities are governed by special regulations. الاستيل...
  [Art   32] sim=0.577  Missing person and absent persons are subject to provisions contained in special laws; in the absence of such special la...

=== After reranking (fetched 20, kept best 5) ===
  [Art    1] rerank=+0.715  Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision...
  [Art   24] rerank=-2.511  The principles of private internatio

## 🔗 Tasks 6 + 7 — RAG prompt engineering

**Why RAG?** Without retrieval, the LLM might invent articles that don't exist. With RAG we:
1. **Retrieve** the most relevant chunks from our vector DB.
2. **Stuff** them into the prompt as numbered context.
3. **Instruct** the LLM to answer *only* from this context and to cite article numbers.

This is *grounding* — the answer becomes traceable and verifiable.

The prompt has four parts:
- **System role** — persona ("legal assistant") and the strict rule ("only use the context")
- **Context block** — each retrieved chunk numbered and labelled with its article number
- **User question** — the original query
- **Instructions tail** — answer format + citation format + fallback when context is insufficient

In [16]:
def build_rag_prompt(question: str, hits: list[dict]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for the LLM call.
    The system prompt is fully bilingual (Arabic + English) and tuned for grounded,
    cited answers."""

    system_prompt = (
        # ---------- English ----------
        "You are a precise, trustworthy legal assistant specialised in the Egyptian Civil Code "
        "(القانون المدني المصري). Follow these rules strictly:\n"
        "1. Answer ONLY using the provided context excerpts. Do NOT use outside knowledge.\n"
        "2. Always cite the relevant article numbers inline, e.g. (Article 12) / (المادة 12).\n"
        "3. If the context does not contain the answer, say exactly:\n"
        "   - EN: 'The provided articles do not contain enough information to answer this question.'\n"
        "   - AR: 'لا تحتوي المواد المقدمة على معلومات كافية للإجابة على هذا السؤال.'\n"
        "4. Never invent articles, numbers, or facts.\n"
        "5. Reply in the SAME language as the user's question (Arabic question -> Arabic answer, "
        "English question -> English answer).\n"
        "6. Be clear, concise, and accurate.\n\n"
        # ---------- Arabic ----------
        "أنت مساعد قانوني دقيق وموثوق متخصص في القانون المدني المصري. التزم بالقواعد التالية بدقة:\n"
        "1. أجب فقط باستخدام المقتطفات المرفقة من السياق، ولا تستخدم أي معرفة خارجية.\n"
        "2. اذكر دائمًا أرقام المواد ذات الصلة داخل الإجابة، مثل (المادة 12).\n"
        "3. إذا لم يحتوِ السياق على الإجابة، فاكتب بالضبط: "
        "'لا تحتوي المواد المقدمة على معلومات كافية للإجابة على هذا السؤال.'\n"
        "4. لا تختلق أي مواد أو أرقام أو معلومات.\n"
        "5. أجب بنفس لغة سؤال المستخدم (سؤال بالعربية ← إجابة بالعربية، سؤال بالإنجليزية ← إجابة بالإنجليزية).\n"
        "6. كن واضحًا ومختصرًا ودقيقًا."
    )

    context_blocks = []
    for i, h in enumerate(hits, 1):
        context_blocks.append(
            f"[{i}] (Article {h['article_number']} | {h['language']})\n{h['text']}"
        )
    context_str = "\n\n".join(context_blocks)

    user_prompt = (
        "Context excerpts from the Egyptian Civil Code / مقتطفات من القانون المدني المصري:\n"
        f"---\n{context_str}\n---\n\n"
        f"Question / السؤال: {question}\n\n"
        "Answer (with article citations) / الإجابة (مع ذكر أرقام المواد):"
    )
    return system_prompt, user_prompt


# Show what a prompt looks like
hits = vector_search("What rules apply when there is no written law?", k=3, language="en")
sys_p, usr_p = build_rag_prompt("What rules apply when there is no written law?", hits)
print("=== SYSTEM PROMPT ===")
print(sys_p)
print("\n=== USER PROMPT (truncated) ===")
print(usr_p[:1200] + "...")


=== SYSTEM PROMPT ===
You are a precise, trustworthy legal assistant specialised in the Egyptian Civil Code (القانون المدني المصري). Follow these rules strictly:
1. Answer ONLY using the provided context excerpts. Do NOT use outside knowledge.
2. Always cite the relevant article numbers inline, e.g. (Article 12) / (المادة 12).
3. If the context does not contain the answer, say exactly:
   - EN: 'The provided articles do not contain enough information to answer this question.'
   - AR: 'لا تحتوي المواد المقدمة على معلومات كافية للإجابة على هذا السؤال.'
4. Never invent articles, numbers, or facts.
5. Reply in the SAME language as the user's question (Arabic question -> Arabic answer, English question -> English answer).
6. Be clear, concise, and accurate.

أنت مساعد قانوني دقيق وموثوق متخصص في القانون المدني المصري. التزم بالقواعد التالية بدقة:
1. أجب فقط باستخدام المقتطفات المرفقة من السياق، ولا تستخدم أي معرفة خارجية.
2. اذكر دائمًا أرقام المواد ذات الصلة داخل الإجابة، مثل (المادة 12).

## 🤖 The LLM — Qwen served via **Ollama**

The LLM turns the retrieved chunks into a **natural-language answer**. Everything is wrapped behind a single `call_llm()` function so nothing else in the pipeline needs to change.

| Backend | Cost | Quality | Needs |
|---|---|---|---|
| `"ollama"` ✅ default | **🆓 Free** | Great Arabic + English | Ollama runtime + a Qwen model (pulled automatically) |
| `"local"` | Free | Good | transformers download from Hugging Face |
| `"anthropic"` | Paid | Excellent | `ANTHROPIC_API_KEY` in Kaggle Secrets |
| `"openai"` | Paid | Excellent | `OPENAI_API_KEY` in Kaggle Secrets |
| `"mock"` | Free | No real answers | Nothing |

### About the Ollama backend

We chat with **`qwen2.5:7b-instruct`** (set in the CONFIG cell as `OLLAMA_MODEL`) — a multilingual instruction-tuned model that handles **Arabic and English** well. The server is started and the model is pulled automatically on the first `call_llm()` call. Use `qwen2.5:3b-instruct` for a lighter/faster option.


In [17]:
# ============================================================================
# LLM backend selector — pick one of: "ollama", "local", "anthropic", "openai", "mock"
# ============================================================================
# DEFAULT: "ollama" — chats with a local Qwen model served by Ollama. FREE, no API key.
LLM_BACKEND = "ollama"

# --- Ollama settings (the main backend) ---
# OLLAMA_MODEL / OLLAMA_HOST come from the CONFIG cell.
OLLAMA_NUM_PREDICT = 512        # max new tokens
OLLAMA_TEMPERATURE = 0.0        # deterministic, grounded legal answers

# --- Local (transformers) backend settings (kept as a fallback) ---
LOCAL_LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
LOCAL_LLM_DEVICE = "cuda"
LOCAL_LLM_MAX_NEW_TOKENS = 400

import subprocess, time as _time

_ollama_ready = False

def _ensure_ollama():
    """Start the Ollama server once and make sure the Qwen model is pulled."""
    global _ollama_ready
    if _ollama_ready:
        return
    os.environ.setdefault("OLLAMA_HOST", OLLAMA_HOST)
    import ollama
    # 1. Start the server in the background if it isn't running yet.
    try:
        ollama.list()
    except Exception:
        print("Starting Ollama server ...")
        subprocess.Popen(["ollama", "serve"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(60):
            try:
                ollama.list()
                break
            except Exception:
                _time.sleep(1)
    # 2. Pull the model if it isn't available locally.
    have = {m.get("model") or m.get("name") or "" for m in ollama.list().get("models", [])}
    if not any(OLLAMA_MODEL in h for h in have):
        print(f"Pulling Ollama model '{OLLAMA_MODEL}' (first time, may take a few minutes) ...")
        ollama.pull(OLLAMA_MODEL)
    print(f"Ollama ready ✓  (model: {OLLAMA_MODEL})")
    _ollama_ready = True


# Lazily loaded transformers pipeline (only used if LLM_BACKEND == "local")
_local_llm_pipe = None

def _get_local_pipe():
    """Load the local LLM once and cache it."""
    global _local_llm_pipe
    if _local_llm_pipe is None:
        from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
        import torch
        print(f"Loading local LLM: {LOCAL_LLM_NAME}  (first call, ~3-4 min for download)")
        tok = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)
        mdl = AutoModelForCausalLM.from_pretrained(
            LOCAL_LLM_NAME,
            torch_dtype=torch.float16 if LOCAL_LLM_DEVICE == "cuda" else torch.float32,
            device_map=LOCAL_LLM_DEVICE,
        )
        _local_llm_pipe = pipeline(
            "text-generation",
            model=mdl,
            tokenizer=tok,
            device=0 if LOCAL_LLM_DEVICE == "cuda" else -1,
        )
        print("Local LLM ready ✓")
    return _local_llm_pipe


def call_llm(system_prompt: str, user_prompt: str) -> str:
    """One function, multiple backends."""

    if LLM_BACKEND == "ollama":
        _ensure_ollama()
        import ollama
        resp = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            options={"temperature": OLLAMA_TEMPERATURE, "num_predict": OLLAMA_NUM_PREDICT},
        )
        return resp["message"]["content"].strip()

    if LLM_BACKEND == "local":
        pipe = _get_local_pipe()
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ]
        out = pipe(
            messages,
            max_new_tokens=LOCAL_LLM_MAX_NEW_TOKENS,
            do_sample=False,
            return_full_text=False,
        )
        return out[0]["generated_text"].strip()

    if LLM_BACKEND == "anthropic":
        import anthropic
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
        client_llm = anthropic.Anthropic(api_key=api_key)
        msg = client_llm.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return msg.content[0].text

    if LLM_BACKEND == "openai":
        from openai import OpenAI
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
        client_llm = OpenAI(api_key=api_key)
        resp = client_llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            max_tokens=1024,
        )
        return resp.choices[0].message.content

    # mock fallback
    return (
        "[MOCK ANSWER — set LLM_BACKEND to 'ollama' for a free real answer]\n"
        f"Retrieved {user_prompt.count('Article')} article(s) as context."
    )

print(f"LLM backend: {LLM_BACKEND}")
if LLM_BACKEND == "ollama":
    print(f"  Ollama model: {OLLAMA_MODEL}")
    print(f"  Host:         {OLLAMA_HOST}")
    print("  (Server starts + model pulls on the first call_llm() call.)")


LLM backend: ollama
  Ollama model: qwen2.5:7b-instruct
  Host:         http://127.0.0.1:11434
  (Server starts + model pulls on the first call_llm() call.)


## 🧠 Keyword extraction → keyword-based vector search

A dedicated, **separate function** that:
1. takes the user's question (Arabic **or** English),
2. asks the **Qwen** model (via Ollama) to extract the key search keywords,
3. returns them as a clean Python **list**,
4. and feeds those keywords into the **vector search** instead of the raw question.


In [18]:
import json as _json
import re as _re

def extract_keywords(question: str, max_keywords: int = 8) -> list[str]:
    """Question (AR or EN) -> Qwen (via Ollama) -> a clean list of search keywords."""
    kw_system = (
        # English
        "You are a keyword extraction engine for a legal search system. "
        "Extract the most important search keywords/phrases from the user's question. "
        "Keep legal terms, entities and topic words; drop stopwords and filler. "
        "Keep each keyword in the SAME language as the question. "
        f"Return ONLY a JSON array of at most {max_keywords} short strings, nothing else.\n\n"
        # Arabic
        "أنت محرّك لاستخراج الكلمات المفتاحية لنظام بحث قانوني. "
        "استخرج أهم الكلمات أو العبارات المفتاحية من سؤال المستخدم، "
        "واحتفظ بالمصطلحات القانونية والكيانات والكلمات الموضوعية واحذف كلمات الوقف والحشو. "
        "اجعل كل كلمة بنفس لغة السؤال. "
        f"أعد فقط مصفوفة JSON تحتوي على {max_keywords} عناصر كحدّ أقصى، دون أي نص آخر."
    )
    kw_user = f"Question / السؤال: {question}\n\nKeywords JSON:"

    raw = call_llm(kw_system, kw_user)

    # Robust parse: grab the JSON array even if the model wraps it in extra text/fences.
    keywords: list[str] = []
    m = _re.search(r"\[.*\]", raw, _re.DOTALL)
    if m:
        try:
            parsed = _json.loads(m.group(0))
            keywords = [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            keywords = []
    # Fallback: split the question if extraction failed.
    if not keywords:
        keywords = [w for w in _re.split(r"[\s،,]+", question) if len(w) > 1]

    # de-duplicate, preserve order
    seen, out = set(), []
    for kw in keywords:
        key = kw.lower()
        if key not in seen:
            seen.add(key)
            out.append(kw)
    return out[:max_keywords]


def vector_search_by_keywords(question: str, k: int = 5, language: str | None = None,
                              backend: str = "chroma") -> dict:
    """question -> keywords (Qwen/Ollama) -> vector search using those keywords.
    Returns {'keywords': [...], 'query': '...', 'hits': [...]}."""
    keywords = extract_keywords(question)
    kw_query = " ".join(keywords) if keywords else question
    if backend == "qdrant":
        hits = vector_search_qdrant(kw_query, k=k, language=language)
    else:
        hits = vector_search(kw_query, k=k, language=language)
    return {"keywords": keywords, "query": kw_query, "hits": hits}


# Demo (Arabic + English)
for q in ["ما هي حقوق المستأجر في عقد الإيجار؟",
          "obligations of the seller in a contract of sale"]:
    res = vector_search_by_keywords(q, k=3)
    print("Question: ", q)
    print("Keywords: ", res["keywords"])
    print("Query used:", res["query"])
    for h in res["hits"]:
        print(f"   [Art {h['article_number']} | sim={h['score']:.3f}] {h['text'][:110]}...")
    print()


Starting Ollama server ...
Pulling Ollama model 'qwen2.5:7b-instruct' (first time, may take a few minutes) ...
Ollama ready ✓  (model: qwen2.5:7b-instruct)
Question:  ما هي حقوق المستأجر في عقد الإيجار؟
Keywords:  ['حقوق المستأجر', 'عقد الإيجار']
Query used: حقوق المستأجر عقد الإيجار
   [Art 558 | sim=0.731] الإيجار عقد يلتزم المؤجر بمقتضاه أن يمكن المستأجر من الانتفاع بشيء معين مدة معينة لقاء أجر معلوم....
   [Art 593 | sim=0.675] للمستأجر حق التنازل عن الإيجار أو الإيجار من الباطن وذلك عن كل ما استأجره أو بعضه ما لم يقض الاتفاق بغير ذلك....
   [Art 589 | sim=0.674] )١ (يكون للمؤجر ، ضمانا لكل حق يثبت له بمقتضى عقد الإيجار ، أي يحبس جميع المنقولات القابلة للحجز الموجود في ال...

Question:  obligations of the seller in a contract of sale
Keywords:  ['obligations', 'seller', 'contract', 'sale']
Query used: obligations seller contract sale
   [Art 418 | sim=0.654] Sale is a contract whereby the vendor binds himself to transfer to the purchaser the ownership of a thing or a...
   [Art 428

## 🚀 Task 9 — End-to-end RAG pipeline

A single function wires everything together:

```
question
   ↓ (1) detect language → set DB filter
   ↓ (2) bi-encoder retrieval → top-N candidates
   ↓ (3) cross-encoder rerank → top-k
   ↓ (4) build grounded prompt
   ↓ (5) call LLM
answer with citations
```

In [19]:
def detect_language(text: str) -> str:
    """Tiny heuristic: if there's any Arabic letter, treat as Arabic; else English."""
    for ch in text:
        if "\u0600" <= ch <= "\u06FF":
            return "ar"
    return "en"


def rag_pipeline(question: str, k: int = 5, candidate_k: int = 20,
                 use_rerank: bool = True, restrict_language: bool = True,
                 use_keywords: bool = True) -> dict:
    """End-to-end RAG.

    Flow: question -> (optional) keyword extraction via Qwen/Ollama -> vector search
    by those keywords -> (optional) rerank -> grounded bilingual prompt -> LLM answer.
    Returns the answer plus the retrieved hits (and the keywords used) for inspection.
    """
    lang = detect_language(question) if restrict_language else None

    # Step 1: turn the question into search keywords (Qwen via Ollama).
    keywords = None
    search_query = question
    if use_keywords:
        keywords = extract_keywords(question)
        if keywords:
            search_query = " ".join(keywords)

    # Step 2: vector search using the keyword query.
    if use_rerank:
        candidates = vector_search(search_query, k=candidate_k, language=lang)
        if candidates:
            # Rerank against the ORIGINAL question for the best relevance signal.
            pairs = [(question, c["text"]) for c in candidates]
            rerank_scores = reranker.predict(pairs)
            for c, s in zip(candidates, rerank_scores):
                c["rerank_score"] = float(s)
            candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
        hits = candidates[:k]
    else:
        hits = vector_search(search_query, k=k, language=lang)

    # Step 3: build the bilingual prompt and call the LLM (Ollama/Qwen by default).
    system_prompt, user_prompt = build_rag_prompt(question, hits)
    answer = call_llm(system_prompt, user_prompt)

    return {
        "question": question,
        "keywords": keywords,
        "search_query": search_query,
        "detected_language": lang,
        "answer": answer,
        "hits": hits,
    }


# Smoke test
result = rag_pipeline("What rules apply when there is no written law?", k=3)
print("Q:", result["question"])
print("Keywords:", result["keywords"])
print("Lang:", result["detected_language"])
print("Answer:", result["answer"])
print("\nRetrieved articles:", [h["article_number"] for h in result["hits"]])


Q: What rules apply when there is no written law?
Keywords: ['written law', 'apply rules']
Lang: en
Answer: إذا لم يكن هناك قانون مكتوب ينطبق، فإن القاضي سيقرر وفقًا للعادة وفي حالة عدم وجود عادة سارية، وفقًا لقواعد الشريعة الإسلامية. وإذا لم تكن هناك قواعد إسلامية، فسيطبق القاضي قواعد العدالة الطبيعية والقواعد العادلة.

(المادة 1) / (Article 1)

Retrieved articles: [1, 24, 190]


# 🧪 Phase C — Measuring quality

## Test queries (gold set)

A small curated set of `(question, expected_articles)` pairs that we'll use for every evaluation method below. Expand this set to get more reliable numbers.

In [20]:
# Curated test set: question + expected article number(s). 
# Tweak these to match your reading of the code — they're starting points.
test_queries = [
    {
        "question": "What rules apply when there is no written law?",
        "expected_articles": [1],
        "language": "en",
    },
    {
        "question": "ما هي الأحكام التي تطبق عند عدم وجود نص تشريعي؟",
        "expected_articles": [1],
        "language": "ar",
    },
    {
        "question": "What are the obligations of the seller in a contract of sale?",
        "expected_articles": list(range(418, 500)),  # the sale section
        "language": "en",
    },
    {
        "question": "حقوق المستأجر في عقد الإيجار",
        "expected_articles": list(range(558, 600)),  # rough lease section
        "language": "ar",
    },
]


def hit_at_k(retrieved_articles: list[int], expected: list[int]) -> int:
    return int(any(a in expected for a in retrieved_articles))


def evaluate(retriever_fn, label: str, k: int = 5):
    hits_total = 0
    for tq in test_queries:
        hits = retriever_fn(tq["question"], k=k, language=tq["language"])
        retrieved = [h["article_number"] for h in hits]
        hit = hit_at_k(retrieved, tq["expected_articles"])
        hits_total += hit
        print(f"  [{label}] Hit@{k}={hit}  retrieved={retrieved[:5]}  q={tq['question'][:60]}")
    print(f"  ==> {label} mean Hit@{k}: {hits_total / len(test_queries):.2f}\n")


print("Retrieval accuracy (mean Hit@5)")
print("-" * 70)
evaluate(vector_search, "Chroma (no rerank)", k=5)
evaluate(vector_search_qdrant, "Qdrant (no rerank)", k=5)
evaluate(lambda q, k, language: vector_search_with_rerank(q, k=k, candidate_k=20, language=language),
         "Chroma + Rerank", k=5)

Retrieval accuracy (mean Hit@5)
----------------------------------------------------------------------
  [Chroma (no rerank)] Hit@5=1  retrieved=[1, 856, 86, 873, 32]  q=What rules apply when there is no written law?
  [Chroma (no rerank)] Hit@5=1  retrieved=[1, 200, 32, 23, 2]  q=ما هي الأحكام التي تطبق عند عدم وجود نص تشريعي؟
  [Chroma (no rerank)] Hit@5=1  retrieved=[418, 431, 1099, 467, 433]  q=What are the obligations of the seller in a contract of sale
  [Chroma (no rerank)] Hit@5=1  retrieved=[558, 589, 593, 572, 582]  q=حقوق المستأجر في عقد الإيجار
  ==> Chroma (no rerank) mean Hit@5: 1.00

  [Qdrant (no rerank)] Hit@5=1  retrieved=[1, 856, 86, 873, 32]  q=What rules apply when there is no written law?
  [Qdrant (no rerank)] Hit@5=1  retrieved=[1, 200, 32, 23, 2]  q=ما هي الأحكام التي تطبق عند عدم وجود نص تشريعي؟
  [Qdrant (no rerank)] Hit@5=1  retrieved=[418, 431, 1099, 467, 433]  q=What are the obligations of the seller in a contract of sale
  [Qdrant (no rerank)] Hit@5=1  re

## 📏 Task 10a — Retrieval metrics (cheap, deterministic)

These judge **the retriever alone** — does it surface the right chunks? They need only the gold set, no LLM calls.

| Metric | What it tells you | Good when… |
|---|---|---|
| **Hit@k** | Did *any* gold article appear in the top-k? (binary) | Quick sanity check |
| **Recall@k** | What fraction of the gold articles did we get back? | The query has multiple correct articles |
| **MRR** (Mean Reciprocal Rank) | How high did the first correct hit appear? `1/rank` | You care about ranking, not just presence |
| **nDCG@k** | Discounted Cumulative Gain — rewards correct hits higher up | The strongest single number for ranking quality |

In [21]:
# ============================================================================
# Retrieval-only metrics
# ============================================================================
import math
from typing import Callable

def hit_at_k_metric(retrieved: list[int], gold: list[int], k: int) -> float:
    """1.0 if any gold article appears in retrieved[:k], else 0.0"""
    return float(any(a in gold for a in retrieved[:k]))

def recall_at_k(retrieved: list[int], gold: list[int], k: int) -> float:
    """Fraction of gold articles found in top-k."""
    if not gold:
        return 0.0
    top = set(retrieved[:k])
    found = sum(1 for a in gold if a in top)
    return found / len(gold)

def mrr(retrieved: list[int], gold: list[int]) -> float:
    """Reciprocal of the rank of the first gold article. 0 if none."""
    for i, a in enumerate(retrieved, 1):
        if a in gold:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: list[int], gold: list[int], k: int) -> float:
    """Normalised Discounted Cumulative Gain (binary relevance, 1 if in gold)."""
    # DCG@k with relevance = 1 for gold articles, 0 otherwise.
    dcg = sum((1.0 / math.log2(i + 2)) for i, a in enumerate(retrieved[:k]) if a in gold)
    # IDCG@k = best possible DCG given how many golds exist.
    n_relevant = min(len(gold), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(n_relevant))
    return (dcg / idcg) if idcg > 0 else 0.0


def evaluate_retriever(retriever_fn: Callable, queries: list[dict], k: int = 5) -> dict:
    """Run a retriever over all test queries and aggregate the four metrics."""
    sums = {"hit@k": 0.0, "recall@k": 0.0, "mrr": 0.0, "ndcg@k": 0.0}
    per_query = []
    for tq in queries:
        hits = retriever_fn(tq["question"], k=k, language=tq.get("language"))
        retrieved = [h["article_number"] for h in hits]
        gold = tq["expected_articles"]
        row = {
            "question":  tq["question"][:60],
            "lang":      tq.get("language", ""),
            "retrieved": retrieved,
            "gold":      gold[:5] + (["..."] if len(gold) > 5 else []),
            "hit@k":     hit_at_k_metric(retrieved, gold, k),
            "recall@k":  recall_at_k(retrieved, gold, k),
            "mrr":       mrr(retrieved, gold),
            "ndcg@k":    ndcg_at_k(retrieved, gold, k),
        }
        per_query.append(row)
        for m in sums:
            sums[m] += row[m]
    n = len(queries)
    averages = {m: sums[m] / n for m in sums}
    return {"averages": averages, "per_query": per_query, "k": k}


# Run it across all three retriever configurations
print("Retrieval evaluation @ k=5")
print("=" * 70)
configs = [
    ("Chroma",        vector_search),
    ("Qdrant",        vector_search_qdrant),
    ("Chroma+Rerank", lambda q, k, language: vector_search_with_rerank(q, k=k, candidate_k=20, language=language)),
]
for name, fn in configs:
    res = evaluate_retriever(fn, test_queries, k=5)
    a = res["averages"]
    print(f"\n{name:<18} Hit@5={a['hit@k']:.2f}  Recall@5={a['recall@k']:.2f}  "
          f"MRR={a['mrr']:.3f}  nDCG@5={a['ndcg@k']:.3f}")


Retrieval evaluation @ k=5

Chroma             Hit@5=1.00  Recall@5=0.54  MRR=1.000  nDCG@5=0.958

Qdrant             Hit@5=1.00  Recall@5=0.54  MRR=1.000  nDCG@5=0.958

Chroma+Rerank      Hit@5=1.00  Recall@5=0.54  MRR=1.000  nDCG@5=0.958


## 🧑‍⚖️ Task 10b — End-to-end evaluation with LLM-as-a-Judge

Retrieval metrics tell us nothing about the **final answer** the user reads. For that, we ask a **strong LLM** to score the generated answer along three axes:

| Axis | Definition | Why it matters |
|---|---|---|
| **Faithfulness** | Does the answer stay grounded in the retrieved context, no hallucinations? | The whole point of RAG |
| **Relevance** | Does the answer actually address the question? | Catches off-topic responses |
| **Completeness** | Does the answer cover the important facts in the context? | Catches partial / lazy answers |

Each axis is 1–5. Why LLM-as-a-Judge? For open-ended legal Q&A there's no single correct phrasing, so BLEU/ROUGE-style metrics correlate weakly with human judgement. LLM-as-a-Judge is the current standard ([Zheng et al. 2023, MT-Bench](https://arxiv.org/abs/2306.05685)) because it scales, doesn't need reference answers, and correlates strongly with human ratings when the judge is sufficiently strong.

> Note: a 3 B local judge is a *weak* judge — scores may be noisy. For more reliable judging, switch `LLM_BACKEND` to `"anthropic"` or `"openai"`. The deterministic retrieval metrics above are unaffected.

In [22]:
# ============================================================================
# LLM-as-a-Judge
# ============================================================================
import json as _json
import random as _random

JUDGE_PROMPT_TEMPLATE = """You are an impartial evaluator of an AI assistant's answer to a legal question about the Egyptian Civil Code.

You will be given:
1. The user's question.
2. The CONTEXT excerpts that were retrieved and shown to the assistant.
3. The assistant's ANSWER.

Score the answer along THREE axes, each on a 1–5 integer scale:

- FAITHFULNESS (1–5): is every factual claim in the answer supported by the CONTEXT? 5 = fully grounded, 1 = mostly hallucinated.
- RELEVANCE (1–5): does the answer address the user's actual question? 5 = directly answers, 1 = off-topic.
- COMPLETENESS (1–5): does the answer cover the important information available in the CONTEXT? 5 = nothing important left out, 1 = misses most of it.

Return ONLY a JSON object with this exact schema, no markdown, no extra text:
{"faithfulness": <int>, "relevance": <int>, "completeness": <int>, "reasoning": "<one sentence>"}

---
QUESTION: {question}

CONTEXT:
{context}

ANSWER:
{answer}
---

JSON output:"""


def _build_context_block(hits: list[dict]) -> str:
    return "\n\n".join(
        f"[{i}] (Article {h['article_number']}) {h['text']}"
        for i, h in enumerate(hits, 1)
    )


def llm_judge(question: str, hits: list[dict], answer: str) -> dict:
    """Ask the judge LLM to score one (question, context, answer) triple."""
    prompt = (JUDGE_PROMPT_TEMPLATE
              .replace("{question}", question)
              .replace("{context}", _build_context_block(hits))
              .replace("{answer}", answer))

    # Mock backend: deterministic-ish scores so the rest of the pipeline can run.
    if LLM_BACKEND == "mock":
        _random.seed(hash(question) % (2**32))
        return {
            "faithfulness": _random.choice([3, 4, 4, 5]),
            "relevance":    _random.choice([3, 4, 4, 5]),
            "completeness": _random.choice([2, 3, 3, 4]),
            "reasoning":    "[MOCK judge] No real LLM call made.",
        }

    raw = call_llm(
        system_prompt="You are a strict evaluator. Output ONLY valid JSON.",
        user_prompt=prompt,
    )

    # Defensive JSON extraction — strip code fences if the model wrapped them.
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
    try:
        return _json.loads(cleaned)
    except Exception:
        # Last resort: find the first {...} block.
        import re as _re
        m = _re.search(r"\{.*?\}", cleaned, _re.S)
        if m:
            try: return _json.loads(m.group(0))
            except Exception: pass
        return {"faithfulness": 0, "relevance": 0, "completeness": 0,
                "reasoning": f"Judge parse failure: {raw[:120]}"}


def evaluate_rag_with_judge(queries: list[dict], retriever_fn: Callable,
                            k: int = 5, candidate_k: int = 20,
                            use_rerank: bool = True) -> dict:
    """End-to-end RAG eval: retrieve -> answer -> judge.

    Returns per-query scores plus averages for the three judge axes.
    Honours `restrict_language` implicitly via tq['language'].
    """
    rows = []
    sums = {"faithfulness": 0.0, "relevance": 0.0, "completeness": 0.0}
    for tq in queries:
        lang = tq.get("language")
        # Retrieve
        if use_rerank:
            cands = retriever_fn(tq["question"], k=candidate_k, language=lang)
            if cands:
                pairs = [(tq["question"], c["text"]) for c in cands]
                scores = reranker.predict(pairs)
                for c, s in zip(cands, scores):
                    c["rerank_score"] = float(s)
                cands.sort(key=lambda c: c["rerank_score"], reverse=True)
            hits = cands[:k]
        else:
            hits = retriever_fn(tq["question"], k=k, language=lang)

        # Answer
        sp, up = build_rag_prompt(tq["question"], hits)
        answer = call_llm(sp, up)

        # Judge
        verdict = llm_judge(tq["question"], hits, answer)
        rows.append({
            "question":     tq["question"][:60],
            "lang":         lang,
            "answer":       answer[:200] + ("..." if len(answer) > 200 else ""),
            "faithfulness": verdict.get("faithfulness", 0),
            "relevance":    verdict.get("relevance", 0),
            "completeness": verdict.get("completeness", 0),
            "reasoning":    verdict.get("reasoning", ""),
        })
        for axis in sums:
            sums[axis] += float(rows[-1][axis])

    n = max(len(queries), 1)
    averages = {axis: sums[axis] / n for axis in sums}
    averages["overall"] = sum(averages.values()) / 3.0
    return {"averages": averages, "per_query": rows}


# Quick smoke test (mock judge unless you set LLM_BACKEND to anthropic/openai)
print(f"\nLLM-as-a-Judge evaluation (backend={LLM_BACKEND})")
print("=" * 70)
judge_res = evaluate_rag_with_judge(test_queries, vector_search, k=5, use_rerank=True)
a = judge_res["averages"]
print(f"Avg Faithfulness: {a['faithfulness']:.2f}/5")
print(f"Avg Relevance:    {a['relevance']:.2f}/5")
print(f"Avg Completeness: {a['completeness']:.2f}/5")
print(f"Overall:          {a['overall']:.2f}/5")



LLM-as-a-Judge evaluation (backend=ollama)
Avg Faithfulness: 4.00/5
Avg Relevance:    4.75/5
Avg Completeness: 3.50/5
Overall:          4.08/5


## ⏱️ Latency benchmark

We isolate three costs per query:
- **embed**: encoding the query with the bi-encoder
- **search**: the vector DB lookup
- **rerank**: scoring `candidate_k` pairs with the cross-encoder

Each measurement runs 5 times — we report the median (more stable than mean for tail latencies).

In [23]:
import time
import statistics

def time_it(fn, *args, repeat=5, **kwargs):
    samples = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        samples.append((time.perf_counter() - t0) * 1000)  # ms
    return statistics.median(samples), out


def measure_pipeline_latency(query: str, k: int = 5, candidate_k: int = 20,
                             language: str | None = None) -> dict:
    # 1) embed query alone
    t_embed, q_emb = time_it(
        lambda: model.encode([query], normalize_embeddings=True),
        repeat=5,
    )
    q_emb_list = q_emb.tolist()

    # 2) Chroma search alone (using the precomputed embedding to isolate search time)
    where = {"language": language} if language else None
    t_search_chroma, _ = time_it(
        lambda: collection.query(query_embeddings=q_emb_list, n_results=candidate_k, where=where),
        repeat=5,
    )

    # 2b) Qdrant search alone
    from qdrant_client.models import Filter, FieldCondition, MatchValue
    flt = Filter(must=[FieldCondition(key="language", match=MatchValue(value=language))]) if language else None
    t_search_qdrant, _ = time_it(
        lambda: qdrant.query_points(collection_name=QDRANT_COLLECTION, query=q_emb_list[0],
                                    limit=candidate_k, query_filter=flt, with_payload=True),
        repeat=5,
    )

    # 3) rerank on candidate_k candidates
    candidates = vector_search(query, k=candidate_k, language=language)
    pairs = [(query, c["text"]) for c in candidates]
    t_rerank, _ = time_it(lambda: reranker.predict(pairs), repeat=3)

    return {
        "embed_ms":          round(t_embed, 1),
        "search_chroma_ms":  round(t_search_chroma, 1),
        "search_qdrant_ms":  round(t_search_qdrant, 1),
        "rerank_ms":         round(t_rerank, 1),
        "total_chroma_norerank_ms": round(t_embed + t_search_chroma, 1),
        "total_chroma_rerank_ms":   round(t_embed + t_search_chroma + t_rerank, 1),
        "total_qdrant_norerank_ms": round(t_embed + t_search_qdrant, 1),
    }


print("Latency on a sample English query (CPU, median of 5 runs, in ms)")
print("-" * 70)
sample = "What rules apply when there is no written law?"
lat = measure_pipeline_latency(sample, k=5, candidate_k=20, language="en")
for k, v in lat.items():
    print(f"  {k:>30s}: {v:>7} ms")

Latency on a sample English query (CPU, median of 5 runs, in ms)
----------------------------------------------------------------------
                        embed_ms:    24.6 ms
                search_chroma_ms:    10.9 ms
                search_qdrant_ms:    42.1 ms
                       rerank_ms:   685.0 ms
        total_chroma_norerank_ms:    35.6 ms
          total_chroma_rerank_ms:   720.6 ms
        total_qdrant_norerank_ms:    66.7 ms


## 🔬 Side-by-side backend comparison

Same query → run through **Chroma**, **Qdrant**, and (if enabled) **Elasticsearch hybrid**. Look at:
- Which articles each backend returns (do they agree?)
- The score scales (cosine, dot, BM25+vector — not directly comparable, but the *ranking* is)
- Latency you saw above

In [24]:
def compare_backends(query: str, k: int = 5, language: str | None = None):
    print(f"Query: {query!r}  |  language={language}  |  top-{k}")
    print("=" * 80)

    backends = [
        ("Chroma",   vector_search),
        ("Qdrant",   vector_search_qdrant),
    ]
    if ENABLE_ES:
        backends.append(("ES (hybrid)", lambda q, k, language: vector_search_es(q, k=k, language=language, hybrid=True)))

    rows = {}
    for name, fn in backends:
        t0 = time.perf_counter()
        hits = fn(query, k=k, language=language)
        elapsed = (time.perf_counter() - t0) * 1000
        rows[name] = (hits, elapsed)

    # Print a table of article numbers per backend
    print(f"{'rank':<6}", end="")
    for name in rows:
        print(f"{name:<28}", end="")
    print()
    print("-" * 80)
    for i in range(k):
        print(f"{i+1:<6}", end="")
        for name, (hits, _) in rows.items():
            if i < len(hits):
                h = hits[i]
                cell = f"Art {h['article_number']} (s={h['score']:.2f})"
            else:
                cell = "-"
            print(f"{cell:<28}", end="")
        print()
    print()
    for name, (_, ms) in rows.items():
        print(f"  {name} took {ms:.1f} ms")


compare_backends("What rules apply when there is no written law?", k=5, language="en")
print()
compare_backends("عقد الإيجار وحقوق المستأجر", k=5, language="ar")

Query: 'What rules apply when there is no written law?'  |  language=en  |  top-5
rank  Chroma                      Qdrant                      
--------------------------------------------------------------------------------
1     Art 1 (s=0.62)              Art 1 (s=0.62)              
2     Art 856 (s=0.62)            Art 856 (s=0.62)            
3     Art 86 (s=0.59)             Art 86 (s=0.59)             
4     Art 873 (s=0.58)            Art 873 (s=0.58)            
5     Art 32 (s=0.58)             Art 32 (s=0.58)             

  Chroma took 36.3 ms
  Qdrant took 66.1 ms

Query: 'عقد الإيجار وحقوق المستأجر'  |  language=ar  |  top-5
rank  Chroma                      Qdrant                      
--------------------------------------------------------------------------------
1     Art 558 (s=0.71)            Art 558 (s=0.71)            
2     Art 572 (s=0.66)            Art 572 (s=0.66)            
3     Art 589 (s=0.66)            Art 589 (s=0.66)            
4     Art 593 (s=0

# 🎨 Phase D — Interactive Gradio UI

## Web interface with four tabs

A complete interactive interface for the whole pipeline:

1. **💬 Ask** — chat-style RAG. Pick the vector backend (Chroma / Qdrant / ES), the language filter, top-k, and whether to rerank. Get a grounded answer with cited articles + a table of retrieved chunks.
2. **🔬 Compare backends** — same query through all backends side-by-side, with agreement count and latency.
3. **📊 Benchmark** — per-stage latency (embed / search / rerank).
4. **🧪 Evaluation** — full evaluation suite: retrieval metrics for the chosen backend, plus LLM-as-a-Judge on the generated answers.

When you run the cell, Gradio prints two URLs:
- a **local** URL (works inside the notebook iframe)
- a **public share link** (`*.gradio.live`) — open it in any browser/phone

The share link expires after 72 h. On Kaggle make sure **Internet is ON**.

In [25]:
import gradio as gr
import pandas as pd

# ---------------------------------------------------------------------------
# Helpers used by the UI — they wrap the functions we already defined above.
# ---------------------------------------------------------------------------

def _hits_to_df(hits):
    """Turn a list of hit-dicts into a Pandas DataFrame for nice display."""
    if not hits:
        return pd.DataFrame(columns=["rank", "article", "lang", "score", "rerank", "text"])
    rows = []
    for i, h in enumerate(hits, 1):
        rows.append({
            "rank": i,
            "article": h["article_number"],
            "lang": h["language"],
            "score": round(h["score"], 4),
            "rerank": round(h.get("rerank_score", float("nan")), 4) if "rerank_score" in h else None,
            "text": h["text"][:300] + ("..." if len(h["text"]) > 300 else ""),
        })
    return pd.DataFrame(rows)


def _retriever_for(backend: str):
    """Map UI backend name -> base retriever function (no rerank)."""
    if backend == "Chroma":
        return vector_search
    if backend == "Qdrant":
        return vector_search_qdrant
    if backend == "Elasticsearch (hybrid)" and ENABLE_ES:
        return lambda q, k, language: vector_search_es(q, k=k, language=language, hybrid=True)
    return vector_search  # fallback


def _retrieve_with_optional_rerank(question, top_k, candidate_k, use_rerank, lang, backend):
    """Unified retrieval that handles rerank for any backend."""
    base = _retriever_for(backend)
    if not use_rerank:
        return base(question, k=top_k, language=lang)
    candidates = base(question, k=candidate_k, language=lang)
    if candidates:
        pairs = [(question, c["text"]) for c in candidates]
        scores = reranker.predict(pairs)
        for c, s in zip(candidates, scores):
            c["rerank_score"] = float(s)
        candidates.sort(key=lambda c: c["rerank_score"], reverse=True)
    return candidates[:top_k]


# ----- Tab 1: Ask --------------------------------------------------------
def ui_ask(question, top_k, candidate_k, use_rerank, lang_filter, backend):
    if not question or not question.strip():
        return "Please type a question.", pd.DataFrame()

    if backend == "Elasticsearch (hybrid)" and not ENABLE_ES:
        return "❌ Elasticsearch is disabled. Set ENABLE_ES=True and re-run the ES cell.", pd.DataFrame()

    lang = (detect_language(question) if lang_filter == "auto"
            else (None if lang_filter == "any" else lang_filter))

    # Extract keywords (Qwen via Ollama), then search the vector DB by those keywords.
    kw = extract_keywords(question)
    search_query = " ".join(kw) if kw else question
    hits = _retrieve_with_optional_rerank(search_query, top_k, candidate_k, use_rerank, lang, backend)

    sys_p, usr_p = build_rag_prompt(question, hits)
    answer = call_llm(sys_p, usr_p)

    cited = sorted({h["article_number"] for h in hits})
    md_answer = (
        f"### Answer\n\n{answer}\n\n"
        f"---\n\n"
        f"**Backend:** {backend} &nbsp;·&nbsp; **Language used:** `{lang or 'any'}` &nbsp;·&nbsp; "
        f"**Rerank:** {'on' if use_rerank else 'off'} &nbsp;·&nbsp; "
        f"**Keywords:** {', '.join(kw) if kw else '—'} &nbsp;·&nbsp; "
        f"**Cited articles:** {', '.join(map(str, cited)) if cited else '—'}"
    )
    return md_answer, _hits_to_df(hits)


# ----- Tab 2: Compare ----------------------------------------------------
def ui_compare(question, top_k, lang_filter):
    if not question or not question.strip():
        empty = pd.DataFrame()
        return "Please type a question.", empty, empty, empty

    lang = (detect_language(question) if lang_filter == "auto"
            else (None if lang_filter == "any" else lang_filter))

    import time
    t0 = time.perf_counter(); chroma_hits = vector_search(question, k=top_k, language=lang); t_ch = (time.perf_counter()-t0)*1000
    t0 = time.perf_counter(); qdrant_hits = vector_search_qdrant(question, k=top_k, language=lang); t_qd = (time.perf_counter()-t0)*1000
    es_hits, t_es = [], None
    if ENABLE_ES:
        t0 = time.perf_counter()
        es_hits = vector_search_es(question, k=top_k, language=lang, hybrid=True)
        t_es = (time.perf_counter()-t0)*1000

    a = {h["article_number"] for h in chroma_hits}
    b = {h["article_number"] for h in qdrant_hits}
    overlap = len(a & b)

    summary = (
        f"### Comparison summary\n\n"
        f"- **Chroma** took **{t_ch:.1f} ms** — returned {len(chroma_hits)} hits\n"
        f"- **Qdrant** took **{t_qd:.1f} ms** — returned {len(qdrant_hits)} hits\n"
        + (f"- **Elasticsearch** took **{t_es:.1f} ms** — returned {len(es_hits)} hits\n" if ENABLE_ES else "")
        + f"\n**Chroma ∩ Qdrant top-{top_k} agreement:** {overlap} / {top_k} articles overlap.\n\n"
        f"_Language filter: `{lang or 'any'}`_"
    )
    return summary, _hits_to_df(chroma_hits), _hits_to_df(qdrant_hits), _hits_to_df(es_hits)


# ----- Tab 3: Benchmark --------------------------------------------------
def ui_benchmark(query):
    if not query or not query.strip():
        query = "What rules apply when there is no written law?"
    import time, statistics
    def med(fn, repeat=5):
        s = []
        for _ in range(repeat):
            t0 = time.perf_counter(); fn(); s.append((time.perf_counter()-t0)*1000)
        return statistics.median(s)

    lang = detect_language(query)
    where = {"language": lang} if lang else None
    t_embed = med(lambda: model.encode([query], normalize_embeddings=True))
    q_emb = model.encode([query], normalize_embeddings=True).tolist()
    t_chroma = med(lambda: collection.query(query_embeddings=q_emb, n_results=20, where=where))
    from qdrant_client.models import Filter, FieldCondition, MatchValue
    flt = Filter(must=[FieldCondition(key="language", match=MatchValue(value=lang))]) if lang else None
    t_qdrant = med(lambda: qdrant.query_points(
        collection_name=QDRANT_COLLECTION, query=q_emb[0], limit=20,
        query_filter=flt, with_payload=True))
    cands = vector_search(query, k=20, language=lang)
    pairs = [(query, c["text"]) for c in cands]
    t_rerank = med(lambda: reranker.predict(pairs), repeat=3)

    lat_df = pd.DataFrame([
        {"stage": "Embed query",           "ms": round(t_embed, 1)},
        {"stage": "Chroma search (k=20)",  "ms": round(t_chroma, 1)},
        {"stage": "Qdrant search (k=20)",  "ms": round(t_qdrant, 1)},
        {"stage": "Rerank (20 pairs)",     "ms": round(t_rerank, 1)},
        {"stage": "TOTAL Chroma + rerank", "ms": round(t_embed + t_chroma + t_rerank, 1)},
        {"stage": "TOTAL Qdrant no rerank","ms": round(t_embed + t_qdrant, 1)},
    ])
    summary = (
        f"### Latency on query: _{query[:80]}{'...' if len(query) > 80 else ''}_\n\n"
        f"Detected language: `{lang}` &nbsp;·&nbsp; 5-run median &nbsp;·&nbsp; CPU"
    )
    return summary, lat_df


# ----- Tab 4: Evaluation -------------------------------------------------
def ui_evaluate(backend, use_rerank, top_k, candidate_k, run_judge):
    """Run full eval suite on test_queries for the chosen backend & options."""
    base = _retriever_for(backend)
    retriever_for_eval = (
        (lambda q, k, language: _retrieve_with_optional_rerank(q, k, candidate_k, True, language, backend))
        if use_rerank else base
    )

    # 1) Retrieval-only metrics
    retr = evaluate_retriever(retriever_for_eval, test_queries, k=top_k)
    a = retr["averages"]
    metrics_df = pd.DataFrame([
        {"metric": "Hit@k",     "value": round(a["hit@k"], 3)},
        {"metric": "Recall@k",  "value": round(a["recall@k"], 3)},
        {"metric": "MRR",       "value": round(a["mrr"], 3)},
        {"metric": "nDCG@k",    "value": round(a["ndcg@k"], 3)},
    ])
    detail_df = pd.DataFrame(retr["per_query"])

    # 2) LLM-as-a-Judge (optional — it costs API calls when not on mock)
    judge_summary = ""
    judge_df = pd.DataFrame()
    if run_judge:
        judge_res = evaluate_rag_with_judge(test_queries, base, k=top_k,
                                            candidate_k=candidate_k, use_rerank=use_rerank)
        ja = judge_res["averages"]
        judge_summary = (
            f"### LLM-as-a-Judge ({LLM_BACKEND} backend)\n\n"
            f"- **Faithfulness:** {ja['faithfulness']:.2f} / 5\n"
            f"- **Relevance:** {ja['relevance']:.2f} / 5\n"
            f"- **Completeness:** {ja['completeness']:.2f} / 5\n"
            f"- **Overall:** {ja['overall']:.2f} / 5\n"
        )
        judge_df = pd.DataFrame(judge_res["per_query"])

    title = (
        f"### Evaluation — {backend}"
        f" {'(+ rerank)' if use_rerank else ''}"
        f" @ k={top_k}\n\n"
        f"Test set: {len(test_queries)} queries."
    )
    return title, metrics_df, detail_df, judge_summary, judge_df


# ---------------------------------------------------------------------------
# Build the UI
# ---------------------------------------------------------------------------

LANG_CHOICES = ["auto", "en", "ar", "any"]
BACKEND_CHOICES = ["Chroma", "Qdrant"] + (["Elasticsearch (hybrid)"] if ENABLE_ES else [])

with gr.Blocks(title="Egyptian Civil Code RAG") as demo_app:
    gr.Markdown(
        "# ⚖️ Egyptian Civil Code — RAG\n"
        "Bilingual (Arabic + English) retrieval over the 1,093 articles. "
        "Pick a vector DB, ask in either language, get a grounded answer with cited articles."
    )

    # ---- Tab 1: Ask ----
    with gr.Tab("💬 Ask"):
        with gr.Row():
            with gr.Column(scale=2):
                q_input = gr.Textbox(
                    label="Your question",
                    placeholder="e.g. What rules apply when there is no written law?  أو  حقوق المستأجر",
                    lines=2,
                )
            with gr.Column(scale=1):
                backend_ask = gr.Radio(BACKEND_CHOICES, value="Chroma",
                                       label="🗄️ Vector database",
                                       info="Which backend to query")
                lang_ask = gr.Dropdown(LANG_CHOICES, value="auto", label="Language filter")
        with gr.Row():
            top_k_ask = gr.Slider(1, 10, value=5, step=1, label="Top-k results")
            cand_k_ask = gr.Slider(5, 50, value=20, step=5, label="Rerank candidate-k")
            rerank_ask = gr.Checkbox(value=True, label="Use cross-encoder reranker")
        ask_btn = gr.Button("🔎 Ask", variant="primary")
        answer_md = gr.Markdown()
        hits_df = gr.Dataframe(
            headers=["rank", "article", "lang", "score", "rerank", "text"],
            label="Retrieved chunks", wrap=True,
        )
        gr.Examples(
            examples=[
                ["What rules apply when there is no written law?", 5, 20, True, "auto", "Chroma"],
                ["ما هي حقوق المستأجر في عقد الإيجار؟",            5, 20, True, "auto", "Qdrant"],
                ["obligations of the seller in a contract of sale", 5, 20, True, "auto", "Chroma"],
            ],
            inputs=[q_input, top_k_ask, cand_k_ask, rerank_ask, lang_ask, backend_ask],
        )
        ask_btn.click(ui_ask,
                      inputs=[q_input, top_k_ask, cand_k_ask, rerank_ask, lang_ask, backend_ask],
                      outputs=[answer_md, hits_df])

    # ---- Tab 2: Compare backends ----
    with gr.Tab("🔬 Compare backends"):
        gr.Markdown("Run the **same query** through every vector backend and see the rankings side-by-side.")
        with gr.Row():
            cmp_q = gr.Textbox(label="Question", lines=2)
            cmp_k = gr.Slider(1, 10, value=5, step=1, label="Top-k")
            cmp_lang = gr.Dropdown(LANG_CHOICES, value="auto", label="Language filter")
        cmp_btn = gr.Button("⚙️ Compare", variant="primary")
        cmp_summary = gr.Markdown()
        with gr.Row():
            chroma_df = gr.Dataframe(label="Chroma", wrap=True)
            qdrant_df = gr.Dataframe(label="Qdrant", wrap=True)
        es_df = gr.Dataframe(label="Elasticsearch (hybrid)", wrap=True, visible=ENABLE_ES)
        cmp_btn.click(ui_compare, inputs=[cmp_q, cmp_k, cmp_lang],
                      outputs=[cmp_summary, chroma_df, qdrant_df, es_df])

    # ---- Tab 3: Benchmark ----
    with gr.Tab("📊 Benchmark"):
        gr.Markdown("Per-stage latency (median of 5 runs).")
        bench_q = gr.Textbox(label="Query to time", value="What rules apply when there is no written law?")
        bench_btn = gr.Button("⏱️ Run benchmark", variant="primary")
        bench_summary = gr.Markdown()
        lat_df_out = gr.Dataframe(label="Latency breakdown (ms)")
        bench_btn.click(ui_benchmark, inputs=bench_q, outputs=[bench_summary, lat_df_out])

    # ---- Tab 4: Evaluation (NEW) ----
    with gr.Tab("🧪 Evaluation"):
        gr.Markdown(
            "Run the **full evaluation suite** on the `test_queries` gold set:\n\n"
            "- **Retrieval metrics**: Hit@k, Recall@k, MRR, nDCG@k (deterministic, free)\n"
            "- **LLM-as-a-Judge**: Faithfulness / Relevance / Completeness on the generated answers "
            "(needs `LLM_BACKEND` ≠ mock for real scores)"
        )
        with gr.Row():
            eval_backend = gr.Radio(BACKEND_CHOICES, value="Chroma",
                                    label="🗄️ Vector database to evaluate")
            eval_rerank = gr.Checkbox(value=True, label="Include reranker")
        with gr.Row():
            eval_topk = gr.Slider(1, 10, value=5, step=1, label="k for metrics")
            eval_candk = gr.Slider(5, 50, value=20, step=5, label="Candidate-k for rerank")
            eval_judge = gr.Checkbox(value=False,
                                     label="🧑‍⚖️ Also run LLM-as-a-Judge (slower, uses API)")
        eval_btn = gr.Button("▶️ Run evaluation", variant="primary")
        eval_title = gr.Markdown()
        eval_metrics = gr.Dataframe(label="Aggregate retrieval metrics")
        eval_detail = gr.Dataframe(label="Per-query retrieval detail", wrap=True)
        eval_judge_summary = gr.Markdown()
        eval_judge_detail = gr.Dataframe(label="Per-query judge scores", wrap=True)
        eval_btn.click(ui_evaluate,
                       inputs=[eval_backend, eval_rerank, eval_topk, eval_candk, eval_judge],
                       outputs=[eval_title, eval_metrics, eval_detail,
                                eval_judge_summary, eval_judge_detail])


# Launch with share=True so we get a public URL.
try:
    demo_app.launch(share=True, debug=False, theme=gr.themes.Soft())
except TypeError:
    demo_app.launch(share=True, debug=False)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f37d1f4845760a4097.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

# ✅ Done

You now have:

- **Three** vector backends populated with the same 2,262 chunks: Chroma (persistent), Qdrant (in-memory), Elasticsearch (optional, hybrid).
- A **cross-encoder reranker** that lifts retrieval precision at the cost of ~hundreds of ms per query.
- A `rag_pipeline()` function that does everything end-to-end: question → retrieve → rerank → prompt → LLM → grounded answer with citations.
- A free local LLM (Qwen 2.5) that needs no API key.
- A **full evaluation suite**: retrieval metrics (Hit@k, Recall@k, MRR, nDCG@k) + LLM-as-a-Judge (Faithfulness, Relevance, Completeness).
- A **Gradio UI** that exposes all of the above.

### Where to go next
- Swap `EMBED_MODEL_NAME` to `intfloat/multilingual-e5-base` (768-dim) for stronger Arabic retrieval.
- Expand `test_queries` from 4 to 50+ for more reliable evaluation numbers.
- Connect a real Elasticsearch cluster (`ENABLE_ES = True`) to enable hybrid search.